# 03 · Batch inference at scale with a shared DynamicBatcher

Score a large corpus of reviews with the trained `review-classifier`. The twist over a plain
fan-out: a **`DynamicBatcher`** shared per replica aggregates records from many concurrent
callers into cost-budgeted batches, so the model runs on full batches instead of one record
at a time.

What this chapter shows:

- **Load once per replica.** `@alru_cache` builds the model + batcher once per container;
  every concurrent `infer_batch` on that replica reuses them.
- **Shared batcher.** `infer_batch` submits its records and awaits the results; the batcher's
  aggregation loop assembles batches across *all* callers and runs `process_fn` once per
  batch. On GPU this is what keeps the device full; here it's CPU sklearn, same shape.
- **Fan-out driver.** The driver chunks the corpus and launches `infer_batch` concurrently on
  the reusable pool, so the batcher always has a full queue.

> To stay self-contained, this notebook trains a quick model in Step 1. In practice you'd
> point `score` at the `review-classifier` Artifact produced by `02_training.ipynb`.

In [ ]:
from pathlib import Path
import asyncio
import pickle

import flyte
import pandas as pd
from async_lru import alru_cache
from flyte.extras import DynamicBatcher
from flyte.io import File

# connect to the cluster
flyte.init_from_config(Path('.flyte') / 'config.yaml')

image = flyte.Image.from_debian_base().with_pip_packages(
    'scikit-learn', 'pandas', 'pyarrow', 'async-lru', 'unionai-reuse>=0.1.9')

## Step 1 · A model to score with

Train a small classifier and keep its `File` — this stands in for the `review-classifier`
Artifact from Chapter 2.

In [ ]:
train_env = flyte.TaskEnvironment(
    name='bi_train',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
)

_POS = ['love it', 'works great', 'excellent quality', 'highly recommend', 'fast and reliable']
_NEG = ['broke immediately', 'terrible support', 'waste of money', 'very disappointed', 'stopped working']

# quick model that stands in for the Chapter 2 review-classifier Artifact
@train_env.task
async def quick_train(n: int = 200) -> File:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    texts = [(_POS if i % 2 == 0 else _NEG)[i % 5] for i in range(n)]
    labels = ['positive' if i % 2 == 0 else 'negative' for i in range(n)]
    model = Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000))])
    model.fit(texts, labels)
    with open('/tmp/model.pkl', 'wb') as f:
        pickle.dump(model, f)
    return await File.from_local('/tmp/model.pkl')

In [ ]:
model_run = flyte.run(quick_train)
model_run.wait()
model_file = model_run.outputs().o0
print('model ->', model_file.path)

## Step 2 · The batcher singleton and the scoring worker

`get_batcher` is `@alru_cache`d, so the model loads and the batcher starts once per replica.
`process_fn` runs the model on a whole aggregated batch and returns results in input order.
`infer_batch` just submits its chunk and awaits the futures.

In [ ]:
worker_env = flyte.TaskEnvironment(
    name='infer_worker',
    image=image,
    resources=flyte.Resources(cpu=2, memory='1Gi'),
    # concurrency>1: several infer_batch calls share a replica, and its batcher
    reusable=flyte.ReusePolicy(replicas=(1, 2), concurrency=8, idle_ttl=120),
)

driver_env = flyte.TaskEnvironment(
    name='infer_driver',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
    depends_on=[worker_env],
)

_UNLABELED = [
    'the battery lasts forever', 'arrived broken and useless', 'does exactly what it says',
    'customer service ignored me', 'beautiful design and fast', 'fell apart after a week',
    'worth every penny', 'never buying this brand again',
]

# load the model and start the batcher ONCE per replica
@alru_cache(maxsize=1)
async def get_batcher(model_path: str) -> DynamicBatcher:
    local = await File.from_existing_remote(model_path).download()
    with open(local, 'rb') as f:
        model = pickle.load(f)

    # runs the model on a whole aggregated batch, results in input order
    async def predict_batch(texts: list[str]) -> list[str]:
        return [str(p) for p in model.predict(texts)]

    # aggregates records from all concurrent callers into cost-budgeted batches
    batcher = DynamicBatcher(
        process_fn=predict_batch, target_batch_cost=64, max_batch_size=64, batch_timeout_s=0.05)
    await batcher.start()
    print(f'loaded model + started batcher (once per replica) from {model_path}')
    return batcher

In [ ]:
@worker_env.task
async def infer_batch(texts: list[str], model: File, chunk_id: int) -> list[str]:
    batcher = await get_batcher(model.path)

    # submit this chunk into the shared batcher
    futures = await batcher.submit_batch(texts)

    results = await asyncio.gather(*futures)

    s = batcher.stats
    print(f'chunk {chunk_id}: scored {len(texts)} '
          f'(batcher: {s.total_batches} batches, avg_size={s.avg_batch_size:.1f}, util={s.utilization:.2f})')
    return [str(r) for r in results]

@driver_env.task(produces_artifacts=True)
async def score(model: File, n_reviews: int = 400, chunk_size: int = 50) -> File:
    corpus = [_UNLABELED[i % len(_UNLABELED)] for i in range(n_reviews)]
    chunks = [corpus[i:i + chunk_size] for i in range(0, len(corpus), chunk_size)]

    # fan out chunks; the batcher fills batches across all of them
    with flyte.group('score-chunks'):
        results = await asyncio.gather(*(infer_batch(c, model, i) for i, c in enumerate(chunks)))

    predicted = [p for chunk in results for p in chunk]
    df = pd.DataFrame({'text': corpus, 'predicted': predicted})
    pos = float((df['predicted'] == 'positive').mean())
    print(f'scored {len(df)} reviews — {pos:.1%} positive')
    
    df.to_parquet('/tmp/review_predictions.parquet', index=False)
    preds_file = await File.from_local('/tmp/review_predictions.parquet')
    # publish the predictions as an Artifact (closes dataset→model→predictions)
    return flyte.artifacts.new(
        preds_file,
        flyte.artifacts.Metadata(
            name='review-predictions',
            description='Batch predictions from the review-classifier',
            attrs={'rows': str(len(df)), 'pct_positive': f'{pos:.3f}'},
        ),
    )

## Step 3 · Fan out the scoring

In [ ]:
# score the whole corpus
score_run = flyte.run(score, model=model_file, n_reviews=400)
print('Run URL:', score_run.url)
score_run.wait()
print('phase:', score_run.phase)
score_run.outputs()

## Recap

- The corpus was scored in parallel, one action per chunk, on a warm worker pool.
- A shared `DynamicBatcher` (loaded once per replica) aggregated records from all concurrent
  callers into cost-budgeted batches — the pattern that keeps a GPU full.
- Predictions are published as the `review-predictions` Artifact, closing the lineage loop.

Next: **`04_serving.ipynb`** — put the model behind a live endpoint and dashboard.